# Amazon - Prime Video Content Performance by Category

In [1]:
import pandas as pd   
import numpy as np 
import polars as pl
from datetime import date

In [2]:
df_content = pd.read_csv('../Data/006/content_views_daily_agg.csv', parse_dates=['view_date'])

pl_content = pl.read_csv('../Data/006/content_views_daily_agg.csv', try_parse_dates=True)

# Pregunta 1

### ¿Cuál es el total acumulado de eventos de visualización para cada categoría de contenido en agosto de 2024? Esta información ayudará al equipo de Prime Video a entender qué géneros de contenido están captando más usuarios durante ese mes

```SQL
SELECT
    category,
    SUM(views) AS total_views
FROM content_views_daily_agg
WHERE view_date BETWEEN '2024-08-01' AND '2024-08-31'
GROUP BY category ;
```

In [10]:
ago = df_content[
    df_content['view_date'].between('2024-08-01','2024-08-31')
].copy()

res = ago.groupby('category').agg(total_views=('views','sum')).reset_index()

In [9]:
res = pl_content.filter(
    pl.col('view_date').is_between(date(2024,8,1),date(2024,8,31))
).group_by('category').agg(
    pl.col('views').sum().alias('total_views')
).with_row_index()

# Pregunta 2

### ¿Qué categorías de contenido acumularon más de 100,000 vistas totales durante el tercer trimestre (Q3) de 2024? Este análisis ayudará a identificar los géneros que están atrayendo un alto volumen de interacción por parte de los espectadores.

```SQL
SELECT
    category,
    SUM(views) AS Total_Views
FROM content_views_daily_agg
WHERE (view_date BETWEEN '2024-07-01' AND '2024-09-30')
GROUP BY category
HAVING SUM(views) > 100000;
```

In [19]:
q3 = df_content[
    df_content['view_date'].between('2024-07-01','2024-09-30')
].copy()

res = q3.groupby('category').agg(
    total_views=('views','sum')
).reset_index()

res = res[
    res['total_views'] > 100000
].copy()

In [22]:
res = (
    pl_content
    .filter(
        pl.col('view_date').is_between(date(2024,7,1),date(2024,9,30))
    )
    .group_by('category')
    .agg(
        pl.col('views').sum().alias('total_views')
    )
    .filter(
        pl.col('total_views') > 100000
    )
)

# Pregunta 3

### En septiembre de 2024, para las categorías de contenido que recibieron más de 500,000 vistas acumuladas, ¿cuál es el total de vistas del mes para cada una de esas categorías?

```SQL
SELECT
    category,
    SUM(views) AS Total_Views
FROM content_views_daily_agg
WHERE (view_date BETWEEN '2024-09-01' AND '2024-09-30')
GROUP BY category
HAVING SUM(views) > 500000
```

In [29]:
sept = df_content[
    df_content['view_date'].between('2024-09-01','2024-09-30')
].copy()

res = sept.groupby('category').agg(total_views=('views','sum'))

res = res[
    res['total_views']>=500000
].reset_index()

In [33]:
res = pl_content.filter(
    pl.col('view_date').is_between(date(2024,9,1),date(2024,9,30))
).group_by('category').agg(
    pl.col('views').sum().alias('total_views')
).filter(
    pl.col('total_views') > 500000
)

In [35]:
res = pl_content.filter(
    (pl.col('view_date').dt.month() == 9) & (pl.col('view_date').dt.year() == 2024)
).group_by('category').agg(
    pl.col('views').sum().alias('total_views')
).filter(
    pl.col('total_views') > 500000
)

res

category,total_views
str,i64
"""Action""",550000
